# Retrieval Evaluation

## What retrieval is

Retrieval is the step where a system takes a query (a question, a search term, a request) and pulls candidate pieces of information out of a larger collection — a corpus — that might help answer it. That's the whole job: given a need, surface the right material from everything that exists.

Concretely, retrieval involves:
- **A corpus**: a pile of documents, chunks, rows, or passages.
- **A query**: the expressed information need.
- **A matching mechanism**: something decides which items in the corpus relate to the query. This might be lexical (matching words/phrases — does "Q3 net income" appear in this chunk?) or semantic (matching meaning via embeddings — "did the company make money" might match "operating profit" even without shared words).
- **A ranking and cutoff**: the system can't return everything, so it orders candidates by estimated relevance and returns the top handful.

That's it. Retrieval doesn't answer anything itself — it just decides *what gets handed downstream*. Everything else (writing a coherent answer from that material) is a separate concern, which is why retrieval can be evaluated entirely on its own terms: did the system surface the right stuff?

**Example:** Query: *"What's our refund policy for international orders?"* Corpus: a company's internal policy wiki. Retrieval's job is to find the chunk(s) describing the international refund policy — not to summarize them, not to phrase a reply, just to locate them.

## What makes retrieval good

Good retrieval isn't one property — it's several, and a system can have some and lack others.

**Relevance.** The returned items actually relate to the specific need, not just the general topic.
- Query: *"What was Tesla's Q3 2023 net income?"* A genuinely relevant result is the chunk containing that specific figure. A *topically adjacent but not relevant* result would be a chunk about Tesla's Q3 2023 stock price reaction — same quarter, same company, wrong fact.

**Coverage / recall.** The fact actually needed exists *somewhere* in the corpus, and retrieval finds it — even if it's phrased differently than the query.
- Query: *"Can I bring my dog to the office?"* Policy doc says: *"Pets are permitted in designated areas on weekdays."* Good retrieval connects "dog" to "pets" and surfaces this passage even though the words don't match exactly.

**Precision.** The good stuff isn't buried under junk. If 8 of the top 10 results are irrelevant, downstream use becomes a needle-in-haystack problem even though the needle is technically "in there somewhere."

**Granularity match.** The unit retrieved (a sentence, a paragraph, a whole document) is sized so the needed fact is intact and usable — not split across a chunk boundary, not so large that the fact drowns in noise.
- Bad granularity: a chunk ends with *"The three eligibility requirements are: 1) employment status,"* and cuts off right before requirements 2 and 3, which live in the next chunk that never gets retrieved.

**Disambiguation.** When a query term has multiple meanings, retrieval picks (or surfaces) the one that matches context, rather than mixing unrelated senses.
- Query: *"Apple's market share in 2023"* in a corpus containing both tech-company documents and a horticulture report on apple cultivation. Good retrieval doesn't return cultivation data.

**Completeness for multi-part needs.** If answering the question genuinely requires combining facts from two different places, retrieval needs to get *both*, not just the easier one.
- Query: *"Which department does the manager of Project Phoenix work in?"* This needs: (1) who manages Project Phoenix, and (2) what department that person is in — likely two different documents. Getting only the first half is an incomplete retrieval, even though it "found something relevant."

**Freshness when it matters.** If the corpus contains both an outdated and a current version of something, retrieval surfaces the current one (or at least doesn't silently prefer the stale one).

## What makes retrieval bad

These are the concrete ways retrieval goes wrong — each is a distinct failure pattern, not just "low quality."

**Topical-but-wrong matches (false relevance).**
Query: *"How do I cancel my subscription?"* Retrieves: a page about how to *upgrade* a subscription. Same general subject, wrong specific need.

**Missed needle (recall failure).**
The answer exists in the corpus in plain language, but retrieval never surfaces it — usually because of vocabulary mismatch.
Query: *"car insurance for teen drivers"* — document says *"automobile coverage for adolescent operators."* A purely lexical retriever can completely miss this if it never learns the words are related.

**Lexical/semantic distractors.**
A document matches the query's words or topic vector closely but is about the wrong entity, time, or context.
Query: *"John Carter's role at Initech"* — corpus has two different "John Carter"s (one at Initech's Boston office, one at its Seattle office, different roles). Retrieval grabs the wrong one because the name match overwhelms the disambiguating detail.

**Staleness.**
Query: *"What's the current PTO policy?"* Corpus has the 2021 policy and the 2024 policy (which superseded it). Retrieval surfaces the 2021 version because it's worded more similarly to the query, or because it's been referenced more often elsewhere in the corpus.

**Redundancy in the top results.**
Five of the top 6 results are near-duplicates of the same paragraph appearing in slightly different documents (common with scraped or syndicated content). This wastes the "slots" available for actually useful, varied information.

**Granularity mismatch (too coarse).**
Query: *"What's the late fee for invoice payments?"* Retrieves: an entire 40-page contract, somewhere within which one sentence mentions the fee. Technically "relevant," practically useless if only the top slice of that document gets used.

**Granularity mismatch (too fine).**
A chunk is split so small it loses its context. Example: a chunk reads *"...this applies only in the cases described above."* — but "the cases described above" was in the *previous* chunk, which wasn't retrieved. The fragment is meaningless on its own.

**Query-side ambiguity collapse.**
Query: *"python performance"* — could mean the programming language or, in a wildlife-monitoring corpus, the speed of a python snake. If the system can't tell from context, and just grabs whichever sense is more common/frequent in the corpus, it silently picks wrong.

**Incomplete multi-hop retrieval.**
Already mentioned above as a "good" property in reverse: when a question needs two linked facts and only one gets retrieved, the system has half the puzzle and doesn't know it's missing the other half.

## How retrieval failures appear in answers

This part matters because retrieval problems are often invisible *until* you look at the output and trace backwards — many issues that look like "the answer is wrong" are actually "the answer never had the right material to work with." Here's how each failure mode tends to surface:

**Recall failure → confident invention.**
If the real fact wasn't retrieved, but something *was* retrieved, an answer often gets generated anyway from whatever's available — and it can sound just as confident as if it were correct. This looks identical to fabrication from the outside, but the root cause is upstream: the right source was never found.
*Example:* Query about a specific refund deadline. The actual policy chunk wasn't retrieved; a general "contact support" chunk was. The output states a deadline that isn't actually in any retrieved source — it's been smoothed over rather than flagged as missing.

**Stale document retrieved → confidently outdated answer.**
The answer is fluent, well-supported by *something*, and simply wrong because the something was superseded.
*Example:* Asked for the current PTO policy, the answer correctly reflects the 2021 document — internally consistent, well-cited, just outdated. Nothing about the answer's phrasing reveals the problem; you'd only catch it by checking which document got pulled.

**Distractor entity retrieved → right shape, wrong substance.**
The answer is structured exactly like a correct one — names, numbers, specifics — but they belong to the wrong person/company/time period.
*Example:* The "John Carter at Initech" mix-up above produces an answer describing a real role at Initech, just the wrong John Carter's role.

**Granularity cutoff → partial or garbled answer.**
The answer trails off, contradicts itself mid-sentence, or states something incomplete because the chunk it came from was truncated.
*Example:* "The eligibility requirements are employment status, and [no further detail provided in the source]" — the source actually lists three requirements, but only the first was in the retrieved chunk.

**Incomplete multi-hop retrieval → answer skips half the question.**
The output addresses the part that had supporting material and quietly drops or guesses at the rest.
*Example:* Asked which department a project's manager belongs to, the answer correctly names the manager but invents or omits the department, because only the "who manages it" document was retrieved.

**Conflicting retrieved chunks → an answer that contradicts itself.**
When both an old and new version of a fact get retrieved together, an answer may blend them — stating two different numbers, or hedging between them in a way that signals the source material itself was inconsistent.
*Example:* "The policy allows 15 days, though it may also be 20 days depending on tenure" when really one figure is current and one is outdated, and both got pulled in.

**Wrong sense of an ambiguous query → answer about the wrong thing entirely.**
*Example:* Asked about "python performance" in a mixed corpus, the answer earnestly discusses runtime optimization techniques when the asker actually meant snake locomotion speed (or vice versa).

**A useful diagnostic habit:** when an answer looks wrong, before assuming the downstream process is at fault, check what was actually retrieved. A surprising number of "wrong answers" trace back to "right answer was sitting in the corpus, but never got pulled" — which is a retrieval problem with a retrieval-shaped fix (better matching, better chunking, better disambiguation), not a downstream one.

# Retrieval Evaluation in Procurement Systems

Before metrics, before scores — retrieval evaluation is really just one question, asked over and over: **"Did the system find the right pieces of text before anyone tried to answer anything?"**

Everything below stays at that layer. We're not touching how good the final written answer sounds — only whether the *raw material* handed to whatever produces that answer was the correct material.

---

## What Retrieval Is

Imagine a procurement knowledge base: tender notices, RFPs, vendor bid submissions, eligibility criteria documents, addenda, compliance certificates, evaluation scoring sheets, contracts.

A person asks something like:

> "What is the bid security amount for Tender T-2024-117?"

**Retrieval** is the step where the system searches that pile of documents and pulls out a small number of relevant chunks — *before* any answer gets written. In our example, a good retrieval step would surface the exact clause from T-2024-117's terms and conditions stating the bid security percentage.

Retrieval evaluation asks: did it find that clause? Did it find the *right* tender's clause, not a similar one from T-2024-116? Did it find *all* the relevant clauses, not just one of three?

It does **not** ask whether the eventual sentence written in response is fluent, well-organized, or persuasive. That's a separate concern entirely.

---

## What Makes Retrieval Good

**1. Topical relevance, not just keyword overlap**
Query: "What insurance must the contractor carry?"
Good retrieval finds the *Insurance Requirements* clause — even if it's titled "Risk Coverage Obligations" and never uses the word "insurance" directly, because the *content* matches the intent.

**2. Completeness — getting all the relevant pieces**
Query: "What are the eligibility criteria for this tender?"
If eligibility is split across "Minimum Annual Turnover" (Section 4.2), "Prior Project Experience" (Section 4.3), and "Valid GST Registration" (Annexure C) — good retrieval surfaces all three, not just the first one it bumps into.

**3. Correct scoping to the right document**
A vendor named "Sundar Infrastructures" might appear in five different tenders. If the question is about *this specific* tender (T-2024-117), good retrieval doesn't accidentally pull that vendor's bid from T-2023-089.

**4. Currency — pulling the latest version**
Tenders often get amended. If Addendum 2 pushed the submission deadline from June 30 to July 15, good retrieval surfaces the addendum, not the original notice.

**5. Sensible chunk granularity**
A clause like:

> "The bid validity period shall be 120 days from the date of opening, unless extended by mutual written consent of both parties."

should come back as one coherent unit. If chunking cuts it at "120 days from the date of opening." and drops the exception clause into a separate chunk that never gets retrieved, the *qualifying condition* — arguably the most important part — disappears.

**6. Disambiguation between similar-sounding things**
"Performance Bank Guarantee" vs. "Bid Security Deposit" vs. "Earnest Money Deposit" are three different financial instruments in procurement, often confused. Good retrieval tells them apart even when a query is loosely worded.

---

## What Makes Retrieval Bad

**1. Semantic drift**
Query: "What's the penalty for late delivery?"
Bad retrieval returns the clause on "penalty for bid withdrawal" instead — different penalty, different trigger, different document — because the embedding model thought "penalty" was the important word and ignored which penalty.

**2. Partial retrieval**
Query: "What documents must be submitted with the bid?"
The tender lists 8 required documents across two pages. Bad retrieval grabs 3 of them. The vendor (or evaluator) now has an incomplete checklist and doesn't know it's incomplete.

**3. Wrong-document contamination**
Two tenders for similar civil works are floated the same month — T-2024-117 (road construction) and T-2024-118 (drainage construction) — by the same department. A query about "the road tender's defect liability period" pulls in T-2024-118's clause because the documents are textually similar and poorly distinguished in the index.

**4. Stale version retrieval**
The original RFP said the contract value ceiling was ₹2 crore. Corrigendum 3 raised it to ₹2.5 crore. Retrieval surfaces the original RFP text because the corrigendum was indexed separately and never got linked back, or because the original chunk simply scores higher on similarity.

**5. Chunk-boundary damage**
A clause spanning a page break:

> "...the Earnest Money Deposit shall be forfeited if the bidder withdraws the bid after the deadline,"
> [— page break —]
> "*except in cases of force majeure as defined in Clause 14.*"

If only the first half is retrieved, the answer that follows will state an absolute forfeiture rule that doesn't actually exist — the exception was real, it just got severed.

**6. Noise drowning signal**
A query about "minimum experience required for bidders" returns 15 chunks: 2 relevant, 13 about unrelated administrative procedures, formatting instructions, or boilerplate disclaimers that happened to share vocabulary. The relevant content is technically *in there*, but buried.

**7. Vocabulary mismatch causing zero retrieval**
The tender document uses "EMD" exclusively. The user asks about "bid security." If the system doesn't recognize these as the same thing, it may retrieve nothing relevant at all — not because the information doesn't exist, but because the query and the document never speak the same language.

**8. Table/structure collapse**
Evaluation criteria are often in tables:

| Criterion | Weight |
|---|---|
| Technical capability | 40% |
| Financial bid | 35% |
| Past performance | 25% |

If chunking flattens this into plain text without preserving row alignment, retrieval might return "40%... 35%... 25%..." with the labels detached, or attached to the wrong row entirely.

---

## How Retrieval Failures Show Up in Answers

This is the diagnostic skill: reading an answer and recognizing *which kind* of retrieval problem produced it — without judging whether the answer is well-written.

**Symptom: confident, specific, but wrong fact**
> "The bid security for this tender is ₹5,00,000."
When it's actually ₹3,50,000 (from a different tender).
→ Usually **wrong-document retrieval**. The answer isn't vague or hedged — it's fully confident, because it's accurately reporting a clause that's just from the wrong tender.

**Symptom: vague, hedged, boilerplate-sounding answer**
> "Generally, bidders are expected to meet certain eligibility requirements as specified in the tender document."
→ Usually a sign that **nothing specific was retrieved**. No real clause came back, so there's nothing concrete to report.

**Symptom: answer addresses only part of a multi-part question**
> Q: "What are the eligibility criteria?"
> A: "Bidders must have a minimum annual turnover of ₹5 crore."
(...but says nothing about the also-required prior-project experience or GST registration)
→ **Partial retrieval**. Only one of several relevant chunks was pulled in.

**Symptom: answer blends details from two different things**
> "Vendor X's bid includes a 3-year warranty and was submitted under Tender T-2024-117."
(when the 3-year warranty actually belongs to Vendor Y's bid in the same tender)
→ **Cross-contamination from noisy retrieval** — overlapping chunks from different vendors' submissions got merged without clear separation.

**Symptom: outdated detail stated as current fact**
> "The submission deadline is June 30, 2026."
(when Addendum 2 moved it to July 15)
→ **Stale version retrieval**. The amendment exists in the corpus but wasn't surfaced.

**Symptom: numbers present but disconnected from meaning**
> "The evaluation weights are 40%, 35%, and 25%."
(without saying which percentage applies to which criterion, or applying them to the wrong criteria)
→ **Table/structure damage** during chunking — the numbers survived, the row-to-label mapping didn't.

**Symptom: "I don't have that information" for something that genuinely exists in the documents**
→ **Zero or failed retrieval**, often a **vocabulary mismatch** — the query's words and the document's words don't overlap enough for the search to connect them.

**Symptom: an absolute rule stated with no exceptions, when exceptions exist**
> "The EMD will be forfeited if the bid is withdrawn early."
(omitting the force-majeure exception that exists two sentences later in the source)
→ **Chunk-boundary severing** — the qualifying clause was split off and never retrieved alongside the main rule.

---

### A worked example, start to finish

**Query:** "Can a vendor submit a bid jointly with another company for Tender T-2024-117?"

**What retrieval pulls back:** A chunk from T-2024-117's general terms saying *"Bids may be submitted by Joint Ventures."* — but the chunk boundary cuts off right before the next sentence: *"...provided the JV agreement is notarized and submitted at least 15 days prior to bid opening."*

**Resulting answer:** "Yes, joint venture bids are permitted for this tender."

**Diagnosis:** Not wrong, but dangerously incomplete — a **chunk-boundary failure**. The retrieved fragment was real and on-topic, which is exactly why this kind of failure is harder to catch than a wrong-document error: everything *sounds* plausible, but a procedural requirement that could disqualify a bid silently vanished.

That last property is worth sitting with: the most dangerous retrieval failures in procurement contexts aren't the ones that produce nonsense — they're the ones that produce something *true but incomplete*, because incomplete-but-confident is exactly what causes a vendor or evaluator to act on faulty information without realizing anything was missing.